<a href="https://colab.research.google.com/github/tensorbytes0202/Deep-learning/blob/main/ResNet(imagenet).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import zipfile

zip_path = "/content/drive/MyDrive/Untitled folder/archive (4).zip"  # apna filename check kar lena
extract_path = "/content/tiny-imagenet"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction Done ✅")

Extraction Done ✅


In [11]:
# =====================
# 1. IMPORTS
# =====================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import time
import matplotlib.pyplot as plt


In [12]:
# =====================
# 2. DATA
# =====================
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [13]:
train_data = datasets.ImageFolder("/content/tiny-imagenet/tiny-imagenet-200/train",transform=transform)
val_data = datasets.ImageFolder("/content/tiny-imagenet/tiny-imagenet-200/val",transform=transform)

In [14]:
train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    num_workers=2,
    pin_memory=True
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
# =====================
# 3. MODEL
# =====================
model = models.resnet18(pretrained=True)

# IMPORTANT: freeze mat karo
for param in model.parameters():
    param.requires_grad = True

model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, 200)
)

model = model.to(device)


In [ ]:
# =====================
# 4. TRAIN
# =====================
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0001)

epochs = 7

train_losses = []
val_accuracies = []

start_time = time.time()

for epoch in range(epochs):

    # TRAIN
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)

    # VALIDATION
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    acc = 100 * correct / total
    val_accuracies.append(acc)

    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}, Val Acc = {acc:.2f}%")

print("Total Training Time:", time.time() - start_time)

Epoch 1: Loss = 2.4477, Val Acc = 0.53%
Epoch 2: Loss = 1.4733, Val Acc = 0.70%
Epoch 3: Loss = 1.2073, Val Acc = 0.60%
Epoch 4: Loss = 1.0348, Val Acc = 0.49%
Epoch 5: Loss = 0.9019, Val Acc = 0.50%
Epoch 6: Loss = 0.7946, Val Acc = 0.51%
Epoch 7: Loss = 0.6899, Val Acc = 0.50%


In [ ]:
# =====================
# 5. GRAPH
# =====================
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(train_losses)
plt.title("Training Loss")

plt.subplot(1,2,2)
plt.plot(val_accuracies)
plt.title("Validation Accuracy")

plt.show()


In [ ]:
torch.save(model, "resnet_model.pth")

In [ ]:
model = models.mobilenet_v2(pretrained=True)

for param in model.parameters():
    param.requires_grad = True

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features, 200
)